In [82]:
import gtsam
import numpy as np
from typing import List

from gtsam import Pose3, StateData
from gtsam.symbol_shorthand import X, V, L, Y, W
import time
import os
import argparse

from collections import defaultdict

In [83]:
TOL_NS = 1_000_000        # 1 ms timestamp matching tolerance
FRAME_STRIDE = 4          # use every 4th non-KF frame to start
MIN_OBS = 3               # min observations per landmark
MAX_REPROJ_PX = 5.0       # init outlier threshold
STATE_DT = 0.5  # GP state spacing [s]
FRAME_DT = 0.05           # EuRoC camera period (20 Hz)

# EuRoC cam0 intrinsics. ORB-SLAM3's EuRoC yaml resizes images 752x480 -> 600x350
# (Camera.newWidth/newHeight), so the pixels in the dumps live in the resized image.
# Pixels are already undistorted (mvKeysUn), so no distortion model here.
SX, SY = 600 / 752, 350 / 480
K = gtsam.Cal3_S2(458.654 * SX, 457.296 * SY, 0.0, 367.215 * SX, 248.375 * SY)

In [84]:
def s_to_ns(s):            # "1403636579.963555574"
    a, b = s.split('.')
    return int(a) * 10**9 + int(b.ljust(9, '0')[:9])

def ns_to_s(ns):           # 1403636579963555574 -> "1403636579.963555574"
    ns = int(ns)
    sign = '-' if ns < 0 else ''
    a, b = divmod(abs(ns), 10**9)
    return f"{sign}{a}.{b:09d}"

def nearest(ts_sorted, q):
    i = np.searchsorted(ts_sorted, q)
    cand = [j for j in (i - 1, i) if 0 <= j < len(ts_sorted)]
    j = min(cand, key=lambda j: abs(int(ts_sorted[j]) - q))
    return j, abs(int(ts_sorted[j]) - q)

In [85]:
def load_f_dataset(path):
    """Parse a TUM file with lines: t_ns tx ty tz qx qy qz qw -> (int64 ns array, [Pose3])."""
    times_ns, poses = [], []
    with open(path) as f:
        for line in f:
            tok = line.split()
            if not tok or tok[0].startswith('#'):
                continue
            times_ns.append(int(tok[0].split('.')[0]))
            tx, ty, tz, qx, qy, qz, qw = map(float, tok[1:8])
            poses.append(Pose3(gtsam.Rot3.Quaternion(qw, qx, qy, qz), gtsam.Point3(tx, ty, tz)))
    return np.array(times_ns, dtype=np.int64), poses

# ---------- load landmarks + keyframe observations ----------
def build_landmark_lookup_dict(path): #, first_frame_ts):
    """ Return dict of Landmark_id and its 3D world Pose"""
    P = {}
    first_frame_obs = []
    for line in open(path):
        s = line.split()
        if s[0] == "P":
            P[int(s[1])] = np.array(list(map(float, s[2:5])))
        elif s[0] == "O" and s[2] == "1403636579.913555622": # TODO: Make this generic for all dataset
            first_frame_obs.append([int(s[1]), float(s[3]), float(s[4])])
    return P, first_frame_obs

def load_per_frame_observations(path, f_times_ns):
    frame_obs = {}
    with open(path) as f:
        for line in f:
            s = line.split()
            temp = [int(s[0]), float(s[2]), float(s[3])]
            f_times_idx, dt = nearest(f_times_ns, s_to_ns(s[1]))
            if dt <= TOL_NS:
                if(f_times_ns[f_times_idx] in frame_obs.keys()):
                    frame_obs[f_times_ns[f_times_idx]].append(temp)
                else:
                    frame_obs[f_times_ns[f_times_idx]] = [temp]
    return frame_obs

## 1. Load and filter (unchanged)

In [86]:
# kf_times_ns, kf_poses = load_kf_dataset("MH_01_data/kf_dataset-MH01_mono.txt")

# Loading all frames with timestamps and  
f_times_ns, f_poses = load_f_dataset("MH_01_data/f_dataset-MH01_mono.txt")
P, first_frame_obs = build_landmark_lookup_dict("MH_01_data/MapPoints_KF_UVobs_dataset-MH01_mono.txt")

# load frame observations
per_frame_obs = load_per_frame_observations("MH_01_data/Frame_obs_MH01_mono.txt", f_times_ns)
per_frame_obs[1403636579913555712] = first_frame_obs
# Other objects
landmarks_in_graph = set()
pending = []
last_GP_state_t = 0

In [71]:
len(f_times_ns)

3679

In [72]:
len(per_frame_obs.keys())

3679

In [ ]:
params = gtsam.ISAM2Params(); params.setRelinearizeThreshold(0.1); params.relinearizeSkip = 1
isam = gtsam.ISAM2(params)

states: List[StateData] = []
full_graph = gtsam.NonlinearFactorGraph()         # copy of every factor, for the final query
landmarks_in_graph = set()
lm_buffer = defaultdict(list)                     # lid -> [(j, u, v)] until it has MIN_OBS
pending = []                                      # measurement frames since the last state
causal = []                                       # newest state's pose right after its update
new_f, new_v = gtsam.NonlinearFactorGraph(), gtsam.Values()
gp_delta_t = 0.5 # every 0.5 seconds, we add a new GP state
Qc = 0.1 * np.ones(6)

In [ ]:
for i in range(len(f_times_ns)):

    if (f_times_ns[i]-last_GP_state_t)/1e9 >= gp_delta_t:  # 0.5 second
        # Add a new GP state to the graph with WNOA prior
        new_state = StateData(f_times_ns[i], f_poses[i])
        states.append(new_state)
        last_GP_state_t = f_times_ns[i]
        print(f"Adding new state at {ns_to_s(f_times_ns[i])} with pose {f_poses[i]}")

        new_v.insert(X(len(states)-1), f_poses[i])
        if len(states) > 1:
            prev_state = states[-2]
            delta_t = (new_state.timestamp - prev_state.timestamp) / 1e9



        # new_state = StateData(f_times_ns[i], f_poses[i])
        # states.append(new_state)
        # last_GP_state_t = f_times_ns[i]
        # print(f"Adding new state at {ns_to_s(f_times_ns[i])} with pose {f_poses[i]}")

        # Add the new state to the ISAM2 graph
        # new_v.insert(X(len(states)-1), f_poses[i])
        # if len(states) > 1:
        #     prev_state = states[-2]
        #     delta_t = (new_state.timestamp - prev_state.timestamp) / 1e9
        #     motion_model = gtsam.BetweenFactorPose3(X(len(states)-2), X(len(states)-1), 
        #                                             Pose3(), gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1, 0.01, 0.01, 0.01])))
        #     new_f.add(motion_model)
        
        # # Update ISAM2 with the new factors and values
        # isam.update(new_f, new_v)
        # new_f.resize(0)
        # new_v.clear()